# 🎙️ Tạo phụ đề tiếng Anh bằng Whisper — Mai Han team

**Cách dùng (chỉ 3 bước):**
1. Bấm menu **Thời gian chạy → Chạy tất cả** (hoặc phím **Ctrl + F9**). Nếu Google hỏi *"Vẫn chạy?"* → bấm **Vẫn chạy**.
2. Chờ nút **"Chọn tệp"** hiện ra ở cuối → chọn **video hoặc file tiếng** (chọn được nhiều file).
3. Chờ chạy xong → máy tự tải về file **`..._EN_whisper.srt`** (nhiều file thì 1 file .zip). Đưa file đó vào ScriptPro như bình thường.

**Bảo mật:** file được tải thẳng vào phiên làm việc tạm (KHÔNG lưu lên Google Drive). Chỉ **bản sao** gửi lên đây bị xoá — **video gốc trên máy bạn giữ nguyên**. Bản sao video xoá ngay sau khi tách tiếng, bản sao tiếng xoá khi xong; cuối cùng xoá sạch thư mục làm việc. Dự án nào agency cấm đưa lên dịch vụ ngoài thì **không dùng công cụ này**.

**Mẹo:** gửi **file tiếng** (.mp3/.wav) nhanh hơn nhiều so với video. Nếu thấy dòng 🐢 "chạy bằng bộ xử lý thường": vào **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU** rồi chạy lại.

In [ ]:
#@title ▶ Bấm nút ▶ bên trái (hoặc Ctrl+F9 để chạy tất cả)
MO_HINH = "large-v3"  #@param ["large-v3", "medium", "small"]
NGON_NGU = "en"  #@param ["en", "auto"]
CORE = '"""Lõi công cụ "Tạo phụ đề tiếng Anh bằng Whisper" (chạy trên Google Colab; cũng chạy được trên máy để thử).\nNotebook `Tao_phu_de_Whisper.ipynb` tải file này từ GitHub rồi gọi `run()`.\n\n- Nhận file TIẾNG hoặc VIDEO (bản sao người dùng tải lên phiên Colab). Video: tự tách tiếng rồi XOÁ BẢN SAO ngay — file gốc trên máy người dùng không bị đụng tới.\n- faster-whisper: thử card đồ hoạ (GPU) trước; lỗi/không có thì tự chuyển sang CPU (chậm hơn) và báo rõ.\n- Chia phụ đề theo chuẩn: tối đa 42 ký tự/dòng, 2 dòng/khung, 6 giây/khung; ngắt ở dấu câu hoặc chỗ ngừng nói.\n- Xong: tự tải file .srt về máy, xoá hết file tạm trong phiên.\n"""\nimport os\nimport re\nimport shutil\nimport subprocess\nimport time\n\nAUDIO_EXT = (".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma")\nMAX_LINE, MAX_LINES, MAX_DUR, MIN_DUR, PAUSE = 42, 2, 6.0, 0.8, 0.6\n\n\ndef fmt_ts(t):\n    t = max(0.0, t); ms = int(round(t * 1000))\n    h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)\n    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"\n\n\ndef wrap_lines(text, width=MAX_LINE):\n    """Chia 1 khung thành tối đa 2 dòng cân đối."""\n    if len(text) <= width: return [text]\n    words = text.split(); best = None\n    for i in range(1, len(words)):\n        a, b = " ".join(words[:i]), " ".join(words[i:])\n        if len(a) <= width and len(b) <= width:\n            score = abs(len(a) - len(b)) - (8 if a.endswith((",", ".", "?", "!")) else 0)\n            if best is None or score < best[0]: best = (score, a, b)\n    return [best[1], best[2]] if best else [text[:width], text[width:]]\n\n\ndef build_cues(words):\n    """words: [(start, end, text)] theo thứ tự. Trả về [(start, end, text)] khung phụ đề."""\n    cues, cur = [], []\n    def flush():\n        if cur:\n            txt = re.sub(r"\\s+", " ", " ".join(w[2].strip() for w in cur)).strip()\n            if txt: cues.append([cur[0][0], cur[-1][1], txt])\n        cur.clear()\n    for w in words:\n        if not w[2].strip(): continue\n        if cur:\n            gap = w[0] - cur[-1][1]\n            new_len = len(" ".join(x[2].strip() for x in cur + [w]))\n            too_long = new_len > MAX_LINE * MAX_LINES or (w[1] - cur[0][0]) > MAX_DUR\n            sentence_end = cur[-1][2].strip().endswith((".", "?", "!")) and new_len > 20\n            if too_long or gap >= PAUSE or sentence_end:\n                flush()\n        cur.append(w)\n        if w[2].strip().endswith((".", "?", "!")) and len(" ".join(x[2].strip() for x in cur)) >= MAX_LINE * MAX_LINES * 0.75:\n            flush()\n    flush()\n    for i, c in enumerate(cues):  # khung quá ngắn -> kéo dài (không chồng khung sau)\n        if c[1] - c[0] < MIN_DUR:\n            nxt = cues[i + 1][0] if i + 1 < len(cues) else c[0] + MIN_DUR\n            c[1] = min(c[0] + MIN_DUR, max(c[1], nxt - 0.05))\n    return [tuple(c) for c in cues]\n\n\ndef to_srt(cues):\n    out = []\n    for i, (s, e, t) in enumerate(cues, 1):\n        out.append(f"{i}\\n{fmt_ts(s)} --> {fmt_ts(e)}\\n" + "\\n".join(wrap_lines(t)) + "\\n")\n    return "\\n".join(out)\n\n\ndef extract_audio(path):\n    """Video/tiếng bất kỳ -> wav 16kHz 1 kênh. Video gốc bị XOÁ ngay sau khi tách."""\n    out = os.path.splitext(path)[0] + "_tieng.wav"\n    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vn", "-ac", "1", "-ar", "16000", out],\n                       capture_output=True, text=True)\n    if r.returncode != 0: raise RuntimeError("Không tách được tiếng: " + (r.stderr or "")[-300:])\n    if not path.lower().endswith(AUDIO_EXT): os.remove(path)  # xoá video ngay\n    return out\n\n\ndef load_model(name="large-v3", log=print):\n    from faster_whisper import WhisperModel\n    try:\n        import ctranslate2\n        if ctranslate2.get_cuda_device_count() > 0:\n            m = WhisperModel(name, device="cuda", compute_type="float16")\n            log("⚡ Đang dùng CARD ĐỒ HOẠ (nhanh).")\n            return m, "gpu"\n    except Exception as e:\n        log(f"⚠️ Không dùng được card đồ hoạ ({str(e)[:120]}) — chuyển sang chạy thường (chậm hơn).")\n    log("🐢 Không có card đồ hoạ — chạy bằng bộ xử lý thường, sẽ CHẬM. Gợi ý: Thời gian chạy → Đổi loại thời gian chạy → GPU T4.")\n    return WhisperModel(name, device="cpu", compute_type="int8"), "cpu"\n\n\ndef read_wav(wav):\n    """Đọc wav 16kHz 1 kênh -> mảng số (không cần thư viện đọc âm thanh riêng, tránh lỗi phiên bản)."""\n    import wave\n    import numpy as np\n    with wave.open(wav, "rb") as f:\n        data = f.readframes(f.getnframes())\n    return np.frombuffer(data, dtype=np.int16).astype(np.float32) / 32768.0\n\n\ndef transcribe_file(model, wav, language="en", log=print):\n    t0 = time.time()\n    segs, info = model.transcribe(read_wav(wav), language=None if language == "auto" else language, beam_size=5,\n                                  vad_filter=True, vad_parameters={"min_silence_duration_ms": 400},\n                                  word_timestamps=True, condition_on_previous_text=False)\n    words, last = [], 0\n    for seg in segs:\n        for w in (seg.words or []):\n            words.append((w.start, w.end, w.word))\n        if info.duration and seg.end - last > 30:\n            last = seg.end; log(f"   … {int(seg.end // 60)}:{int(seg.end % 60):02d} / {int(info.duration // 60)}:{int(info.duration % 60):02d}")\n    log(f"   ✔ xong trong {int(time.time() - t0)} giây")\n    return build_cues(words)\n\n\ndef run(paths, model_name="large-v3", language="en", log=print):\n    """Chạy cho danh sách file đã tải vào phiên. Trả về danh sách file .srt đã tạo."""\n    model, dev = load_model(model_name, log)\n    made = []\n    for p in paths:\n        name = os.path.basename(p); log(f"\\n🎬 {name}")\n        wav = None\n        try:\n            wav = extract_audio(p); log("   ✔ đã tách tiếng" + ("" if p.lower().endswith(AUDIO_EXT) else " và xoá bản sao video trên Colab (video gốc trên máy bạn vẫn còn)"))\n            cues = transcribe_file(model, wav, language, log)\n            srt = os.path.splitext(p)[0] + "_EN_whisper.srt"\n            with open(srt, "w", encoding="utf-8") as f: f.write(to_srt(cues))\n            made.append(srt); log(f"   ✔ {len(cues)} khung phụ đề → {os.path.basename(srt)}")\n        except Exception as e:\n            log(f"   ❌ Lỗi: {e}")\n        finally:  # dù thành công hay lỗi: xoá video/tiếng gốc và file tiếng tạm (bảo mật)\n            for f in (p, wav):\n                if f and os.path.exists(f) and not f.lower().endswith(".srt"):\n                    try: os.remove(f)\n                    except OSError: pass\n    return made\n\n\ndef cleanup(folder):\n    """Xoá sạch thư mục làm việc trong phiên Colab (ra khỏi thư mục trước, vì không xoá được thư mục đang đứng)."""\n    try: os.chdir(os.path.dirname(os.path.abspath(folder)))\n    except OSError: pass\n    shutil.rmtree(folder, ignore_errors=True)\n'
import os, sys, subprocess, time, zipfile
open("/content/whisper_colab_core.py", "w", encoding="utf-8").write(CORE)  # phần lõi (nhúng sẵn ở trên)
print("⏳ Đang cài đặt (1–2 phút, chỉ lần đầu mỗi phiên)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=False)
sys.path.insert(0, "/content")
import importlib, whisper_colab_core as W; importlib.reload(W)
from google.colab import files
WORK = "/content/mh_work"; os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print("\n📂 Chọn file VIDEO hoặc TIẾNG (chọn được nhiều file một lúc):")
up = files.upload()
paths = [os.path.join(WORK, n) for n in up]
if not paths:
    print("Chưa chọn file nào. Bấm ▶ để chạy lại.")
else:
    t0 = time.time()
    srts = W.run(paths, MO_HINH, NGON_NGU)
    if srts:
        if len(srts) == 1: out = srts[0]
        else:
            out = os.path.join(WORK, "PhuDe_Whisper.zip")
            with zipfile.ZipFile(out, "w") as z:
                for s in srts: z.write(s, os.path.basename(s))
        print(f"\n✅ XONG {len(srts)}/{len(paths)} file trong {int(time.time() - t0)} giây — đang tải về máy...")
        files.download(out); time.sleep(8)
    W.cleanup(WORK); print("🧹 Đã xoá sạch các BẢN SAO trong phiên Colab (file gốc trên máy bạn vẫn còn nguyên).")